In [2]:
import torch
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU", torch.cuda.get_device_name(0))

In [3]:

import pandas as pd
import numpy as np

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")

print("Path to dataset files:", path)


Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Path to dataset files: /kaggle/input/enron-email-dataset


In [5]:
import os
df = pd.read_csv(os.path.join(path, "emails.csv"))

In [5]:

df.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 10)

In [7]:
df.head()

,file,message
0,allen-p/_sent_mail/1.,"Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>\nDate: Mon, 14 May 2001 16:39:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: tim.belden@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Tim Belden <Tim Belden/Enron@EnronXGate>\nX-cc: \nX-bcc: \nX-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nHere is our forecast\n\n"
1,allen-p/_sent_mail/10.,"Message-ID: <15464986.1075855378456.JavaMail.evans@thyme>\nDate: Fri, 4 May 2001 13:51:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: john.lavorato@enron.com\nSubject: Re:\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: John J Lavorato <John J Lavorato/ENRON@enronXgate@ENRON>\nX-cc: \nX-bcc: \nX-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail\nX-Origin: Allen-P\nX-FileName: pallen (Non-Privileged).pst\n\nTraveling to have a business meeting takes the fun out of the trip. Especially if you have to prepare a presentation. I would suggest holding the business plan meetings here then take a trip without any formal business meetings. I would even try and get some honest opinions on whether a trip is even desired or necessary.\n\nAs far as the business meetings, I think it would be more productive to try and stimulate discussions across the different groups about what is working and what is not. Too often the presenter speaks and the others are quiet just waiting for their turn. The meetings might be better if held in a round table discussion format. \n\nMy suggestion for where to go is Austin. Play golf and rent a ski boat and jet ski's. Flying somewhere takes too much time.\n"
2,allen-p/_sent_mail/100.,"Message-ID: <24216240.1075855687451.JavaMail.evans@thyme>\nDate: Wed, 18 Oct 2000 03:00:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: leah.arsdall@enron.com\nSubject: Re: test\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Leah Van Arsdall\nX-cc: \nX-bcc: \nX-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail\nX-Origin: Allen-P\nX-FileName: pallen.nsf\n\ntest successful. way to go!!!"
3,allen-p/_sent_mail/1000.,"Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>\nDate: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: randall.gay@enron.com\nSubject: \nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Randall L Gay\nX-cc: \nX-bcc: \nX-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail\nX-Origin: Allen-P\nX-FileName: pallen.nsf\n\nRandy,\n\n Can you send me a schedule of the salary and level of everyone in the \nscheduling group. Plus your thoughts on any changes that need to be made. \n(Patti S for example)\n\nPhillip"
4,allen-p/_sent_mail/1001.,"Message-ID: <30922949.1075863688243.JavaMail.evans@thyme>\nDate: Thu, 31 Aug 2000 05:07:00 -0700 (PDT)\nFrom: phillip.allen@enron.com\nTo: greg.piper@enron.com\nSubject: Re: Hello\nMime-Version: 1.0\nContent-Type: text/plain; charset=us-ascii\nContent-Transfer-Encoding: 7bit\nX-From: Phillip K Allen\nX-To: Greg Piper\nX-cc: \nX-bcc: \nX-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail\nX-Origin: Allen-P\nX-FileName: pallen.nsf\n\nLet's shoot for Tuesday at 11:45."


In [6]:
from email import message_from_string

def parse_email_id(raw_msg):
    try:
        msg = message_from_string(raw_msg)

        #extract the headers for the message column
        msg_id = msg.get("Message-ID")
        sender = msg.get("From")
        receiver = msg.get("To")
        date = msg.get("Date")
        subject = msg.get("Subject")
        body = ""
        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == "text/plain":
                    body += part.get_payload(decode=True).decode(errors="ignore")
        else:
            payload = msg.get_payload(decode=True)
            if payload:
                body = payload.decode(errors="ignore")

        return msg_id, sender, receiver, date, subject, body.strip()

    except Exception:
        return None, None, None, None, None, None

#apply parsing
parsed = df["message"].apply(parse_email_id)

df_parsed = pd.DataFrame(parsed.tolist(),
                         columns=["message_id", "from", "to", "date", "subject", "body"])

# keep original file column
df_final = pd.concat([df["file"], df_parsed], axis=1)

# Convert date
df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")

df_final.head(3)


/tmp/ipython-input-3704971510.py:38: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")
/tmp/ipython-input-3704971510.py:38: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_final["date"] = pd.to_datetime(df_final["date"], errors="coerce")


,file,message_id,from,to,date,subject,body
0,allen-p/_sent_mail/1.,<18782981.1075855378110.JavaMail.evans@thyme>,phillip.allen@enron.com,tim.belden@enron.com,2001-05-14 16:39:00-07:00,,Here is our forecast
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,john.lavorato@enron.com,2001-05-04 13:51:00-07:00,Re:,Traveling to have a business meeting takes the...
2,allen-p/_sent_mail/100.,<24216240.1075855687451.JavaMail.evans@thyme>,phillip.allen@enron.com,leah.arsdall@enron.com,2000-10-18 03:00:00-07:00,Re: test,test successful. way to go!!!


In [9]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 10)

In [10]:
# Install Gemini client
!pip install google-generativeai --quiet

In [7]:
import google.generativeai as genai
import os

# Initialize Gemini client
# To use the Gemini API, you'll need an API key. If you don't already have one, create a key in Google AI Studio.
# In Colab, add the key to the secrets manager under the "🔑" in the left panel. Give it the name `GEMINI_API_KEY`.
# Then pass the key to the SDK:
from google.colab import userdata
try:
  GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
  genai.configure(api_key=GOOGLE_API_KEY)
except Exception as e:
  print(f"Error configuring Gemini API: {e}")
  print("Please make sure you have added your GEMINI_API_KEY to Colab secrets.")

def detect_meeting_intent(email_body):
    prompt = f"Determine if the following email mentions about an upcoming meeting, scheduling an upcoming meeting or discussing or connecting:\n\n{email_body}\n\nAnswer with 'Yes' or 'No'."

    try:
        # Initialize the Generative Model
        gemini_model = genai.GenerativeModel('gemini-2.5-flash')
        response = gemini_model.generate_content(
            contents=[{"parts": [{"text": prompt}]}]
        )
        answer = response.text.strip().lower()
        return answer
    except Exception as e:
        print("Error:", e)
        return 0

In [12]:
#Subset df_final if body has "meet" in it
df_final_meet = df_final[df_final['body'].str.contains('meet')]

In [13]:
df_final_meet.head()

,file,message_id,from,to,date,subject,body
1,allen-p/_sent_mail/10.,<15464986.1075855378456.JavaMail.evans@thyme>,phillip.allen@enron.com,john.lavorato@enron.com,2001-05-04 13:51:00-07:00,Re:,Traveling to have a business meeting takes the...
12,allen-p/_sent_mail/105.,<13116875.1075855687561.JavaMail.evans@thyme>,phillip.allen@enron.com,keith.holst@enron.com,2000-10-09 07:16:00-07:00,Consolidated positions: Issues & To Do list,---------------------- Forwarded by Phillip K ...
13,allen-p/_sent_mail/106.,<2707340.1075855687584.JavaMail.evans@thyme>,phillip.allen@enron.com,keith.holst@enron.com,2000-10-09 07:00:00-07:00,Consolidated positions: Issues & To Do list,---------------------- Forwarded by Phillip K ...
19,allen-p/_sent_mail/111.,<29177675.1075855687692.JavaMail.evans@thyme>,phillip.allen@enron.com,ina.rangel@enron.com,2000-10-03 09:15:00-07:00,Meeting re: Storage Strategies in the West,---------------------- Forwarded by Phillip K ...
27,allen-p/_sent_mail/119.,<10523086.1075855687873.JavaMail.evans@thyme>,phillip.allen@enron.com,ina.rangel@enron.com,2000-09-26 07:01:00-07:00,,---------------------- Forwarded by Phillip K ...


In [14]:
# Optionally process a subset first
test_df = df_final_meet.sample(10)  # sample 500 emails for testing

# Apply Gemini
test_df["meeting_intent"] = test_df["body"].apply(detect_meeting_intent)

# Save results
test_df.to_csv("enron_meeting_intent_labeled_sample.csv", index=False)
print("Labeled dataset saved!")


Labeled dataset saved!


In [15]:
# Optionally process a subset first
test_df = df_final.sample(10)  # sample 10 emails for testing

# Apply Gemini
test_df["meeting_intent"] = test_df["body"].apply(detect_meeting_intent)

# Save results
test_df.to_csv("enron_meeting_intent_labeled_sample2.csv", index=False)
print("Labeled dataset saved!")


Labeled dataset saved!


In [19]:
test_df.head(10)

,file,message_id,from,to,date,subject,body,meeting_intent
505717,white-s/meetings/136.,<14741590.1075858777461.JavaMail.evans@thyme>,stacey.white@enron.com,None,2001-06-26 15:32:00-07:00,happy hour with London,CALENDAR ENTRY:\tINVITATION\n\nDescription:\n\...,yes
236222,kean-s/archiving/untitled/8941.,<11824635.1075850040302.JavaMail.evans@thyme>,james.steffes@enron.com,"richard.shapiro@enron.com, steven.kean@enron.com",2001-06-20 00:30:00-07:00,DWR Stranded costs: $21 billion,"As you can see below, the DWR deals are horrib...",yes
48279,cash-m/all_documents/461.,<24416121.1075860487655.JavaMail.evans@thyme>,michelle.cash@enron.com,catherine.huynh@enron.com,2000-10-13 10:12:00-07:00,Re: RW separations,"Cathy, what do we want to do with regard to Pa...",yes
415,allen-p/_sent_mail/477.,<31184462.1075855726546.JavaMail.evans@thyme>,phillip.allen@enron.com,cbpres@austin.rr.com,2001-02-23 08:24:00-08:00,Re: Genesis Plant Tour,"George,\n\nI can take a day off the week I get...",yes
227242,kaminski-v/var/110.,<20153605.1075856643243.JavaMail.evans@thyme>,tanya.tamarchenko@enron.com,andreas.barschkis@mgusa.com,2000-07-06 09:57:00-07:00,Re: ORACLE tables which contain the position a...,"Andreas,\nas we discussed today, I am sending ...",yes
477317,taylor-m/all_documents/8411.,<28224156.1075860210862.JavaMail.evans@thyme>,harry.collins@enron.com,mark.taylor@enron.com,2001-05-25 10:03:00-07:00,Re: Product Type approval needed (CAN Newsprin...,Good to go!\n\n\n\n\tMark Taylor\n\t05/25/2001...,no
454760,steffes-j/inbox/322.,<22360809.1075861635048.JavaMail.evans@thyme>,harry.kingerski@enron.com,"don.black@enron.com, teresa.mihalik@enron.com,...",2001-11-20 13:08:43-08:00,Peoples Gas - IL,Peoples Gas has backed down from their request...,no
197762,jones-t/sent/5813.,<17514293.1075847551076.JavaMail.evans@thyme>,tana.jones@enron.com,"tom.moran@enron.com, walter.guidroz@enron.com",2001-03-06 07:22:00-08:00,MidAmerican Energy Company,Is there any reason why this counterparty has ...,yes
481421,taylor-m/notes_inbox/1907.,<18209470.1075860036663.JavaMail.evans@thyme>,exchangeinfo@nymex.com,mark.taylor@enron.com,2000-12-07 08:10:00-08:00,(00-419) CFTC Approves Amendments to Natural G...,TO: All New York Mercantile Exchange Members\n...,no
458268,stokley-c/chris_stokley/projects/sqmd/25.,<26926052.1075858516689.JavaMail.evans@thyme>,ken.lewchuk@enron.com,chris.stokley@enron.com,2001-07-30 09:45:11-07:00,ISO Audit,"Chris,\n\nI submitted a WIP last week for 3 ot...",yes


In [21]:
pd.set_option('display.max_colwidth', None)
print(test_df[['subject','body','meeting_intent']])

                                                                                             subject  \
505717                                                                        happy hour with London   
236222                                                               DWR Stranded costs: $21 billion   
48279                                                                             Re: RW separations   
415                                                                           Re: Genesis Plant Tour   
227242  Re: ORACLE tables which contain the position and curves information\n for Enron's VAR system   
477317                                          Re: Product Type approval needed (CAN Newsprint Phy)   
454760                                                                              Peoples Gas - IL   
197762                                                                    MidAmerican Energy Company   
481421                            (00-419) CFTC Approves Amendme

In [22]:
subset = test_df[['subject','body','meeting_intent']]
subset.head(10)

subject  \
505717                                                                        happy hour with London   
236222                                                               DWR Stranded costs: $21 billion   
48279                                                                             Re: RW separations   
415                                                                           Re: Genesis Plant Tour   
227242  Re: ORACLE tables which contain the position and curves information\n for Enron's VAR system   
477317                                          Re: Product Type approval needed (CAN Newsprint Phy)   
454760                                                                              Peoples Gas - IL   
197762                                                                    MidAmerican Energy Company   
481421                            (00-419) CFTC Approves Amendments to Natural Gas Price Limit Rules   
458268                                                                                     ISO Audit   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    